# Feature Engineering - RFM Analysis

RFM stands for Recency, Frequency, Monetary - three metrics to understand customer value.

- **Recency (R):** Days since last purchase
- **Frequency (F):** Number of times they bought
- **Monetary (M):** Total amount spent

We'll create these features from transaction data.

In [ ]:
# Load the cleaned data

import pandas as pd
import numpy as np
from datetime import datetime

# Load cleaned data from previous step
df = pd.read_csv('../data/processed/online_retail_cleaned.csv')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"Loaded {len(df):,} transactions")
print(f"Customers: {df['CustomerID'].nunique():,}")

df.head()

In [ ]:
# Step 1: Calculate RECENCY

# Reference date - latest in our data
latest_date = df['InvoiceDate'].max()

# Last purchase date for each customer
last_purchase = df.groupby('CustomerID')['InvoiceDate'].max().reset_index()
last_purchase.columns = ['CustomerID', 'LastPurchaseDate']

# Days since last purchase
last_purchase['Recency'] = (latest_date - last_purchase['LastPurchaseDate']).dt.days

print(f"Average Recency: {last_purchase['Recency'].mean():.0f} days")
last_purchase.head()

In [ ]:
# Step 2: Calculate FREQUENCY

frequency = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
frequency.columns = ['CustomerID', 'Frequency']

print(f"Average Frequency: {frequency['Frequency'].mean():.1f} orders")
frequency.head()

In [ ]:
# Step 3: Calculate MONETARY

monetary = df.groupby('CustomerID')['Amount'].sum().reset_index()
monetary.columns = ['CustomerID', 'Monetary']

print(f"Average Monetary: £{monetary['Monetary'].mean():.2f}")
monetary.head()

In [ ]:
# Combine R, F, M into one table

# Start with recency
rfm = last_purchase[['CustomerID', 'Recency']].copy()

# Add frequency and monetary
rfm = rfm.merge(frequency, on='CustomerID')
rfm = rfm.merge(monetary, on='CustomerID')

print(f"RFM table: {len(rfm)} customers")
print("\nSummary:")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

rfm.head()

In [ ]:
# Step 5: Create RFM Scores (1-4 scale)

# Recency: lower is better (4 = most recent, 1 = oldest)
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=4, labels=[4, 3, 2, 1], duplicates='drop')

# Frequency: higher is better (4 = most frequent, 1 = least)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1, 2, 3, 4], duplicates='drop')

# Monetary: higher is better (4 = highest spender, 1 = lowest)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=4, labels=[1, 2, 3, 4], duplicates='drop')

# Convert to int
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)
rfm['M_Score'] = rfm['M_Score'].astype(int)

print("RFM Scores created")
rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score']].head()

In [ ]:
# Step 6: Save RFM features

# Keep only the RFM columns for clustering
rfm_for_clustering = rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary']].copy()

rfm_for_clustering.to_csv('../data/processed/rfm_features.csv', index=False)

print("✓ RFM features saved")
print("✓ Ready for clustering")